# Stage one: a success checker, and whether it can be trusted

Reviewer 3's seventh comment asks for an intervention experiment. Such an experiment
measures end-to-end task success before and after intervening, and the benchmark has no
automatic success checker: `benchmark.py` generates task strings and nothing else. So
the first question is not whether an intervention helps. It is whether success can be
decided programmatically on this benchmark at all.

It can, in principle, because the correct answer is derivable for all eight templates
from the released knowledge base. T0, T1, T5 and T6 are a lookup or an arithmetic
result. T2, T4 and T7 ask about the six cities whose population is absent by design, so
the correct answer there is a refusal. T3 is mixed: the capital is available and the
country's population is not.

The checker is written here and then **validated against the 200 human labels**, which
give a binary reference: 72 successes against 128 failures. The validation is the point
of this notebook. If the checker disagrees with the human reference too often, the
intervention experiment cannot rest on it, and that is a result worth having before any
GPU time is spent.

One thing is expected in advance and is not a defect. The checker answers "is the final
answer correct", while the human label answers "what is the first observable fault".
A trajectory can contradict itself mid-chain and still end on the right number; the
human calls that F6 and the checker calls it a success. Those disagreements are listed
rather than hidden, because they tell us what the checker does and does not measure.

CPU only. No model is loaded.


In [ ]:
import ast, json, re, unicodedata, urllib.request
from collections import Counter, defaultdict

BASE = ("https://raw.githubusercontent.com/achenna-source/"
        "icv-agent-observability/neucom-d-26-17269-r1/")
fetch = lambda p: urllib.request.urlopen(BASE + p, timeout=90).read().decode("utf-8")

rows = [json.loads(l) for l in fetch("data/trajectories_200.jsonl").splitlines()
        if l.strip()]
_kb = json.loads(fetch("data/knowledge_base.json"))
KB, ABSENT = _kb["facts"], _kb["populations_absent_by_design"]
steps_of = lambda r: (ast.literal_eval(r["trajectory"])
                      if isinstance(r["trajectory"], str) else r["trajectory"])

print("trajectories %d | facts %d | cities absent by design %d"
      % (len(rows), len(KB), len(ABSENT)))
print("templates: %s" % dict(Counter(r["template"] for r in rows)))
print("human labels: %s" % dict(Counter(r["label_human"] for r in rows).most_common()))


## The checker

Deterministic, template-aware, and written out in full so that it can be read before it
is used. Three primitives: text normalisation that drops accents and case, extraction of
digit runs with separators removed, and a refusal test over a fixed keyword list.

The rule for the three absent-city templates deserves a note. A refusal is correct, but
a refusal that also states a population for the absent city is not. The test therefore
requires a refusal marker **and** that every long number in the answer be a value the
store actually holds, so that an invented figure fails even when it is wrapped in
refusal wording.


In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s or "").lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9 ]+", " ", s)

def numbers(s):
    # Suites de trois chiffres ou plus, separateurs retires, en entiers.
    out = set()
    for m in re.finditer(r"\d[\d  .,]{2,}\d|\d{3,}", str(s or "")):
        d = re.sub(r"\D", "", m.group())
        if d:
            out.add(int(d))
    return out

REFUSAL = ("non trouve", "pas trouve", "introuvab", "non disponible",
           "pas disponible", "indisponible", "ne figure pas", "aucune information",
           "aucune donnee", "pas dans la base", "inconnu", "impossible",
           "je ne peux pas", "pas de donnee", "information non", "n est pas dans")

def refuses(text):
    t = norm(text)
    return any(k in t for k in REFUSAL)

KB_VALUES = {int(v) for v in KB.values() if str(v).isdigit()}
POP = {k.split(" ", 1)[1]: int(v) for k, v in KB.items()
       if k.startswith("population ")}
CAP = {k.split(" ", 1)[1]: v for k, v in KB.items() if k.startswith("capitale ")}

def final_answer(steps):
    for s in reversed(steps):
        if str(s.get("action")).strip().lower() == "finish":
            v = s.get("input")
            return "" if v is None or str(v).strip() in ("", "None") else str(v)
    return ""

def expected(rec):
    # (kind, payload) : ce qu'une reponse correcte doit contenir pour cette tache.
    task, tpl = rec["task"], rec["template"]
    t = norm(task)
    if tpl == "T0":
        m = re.search(r"capitale de ([a-z]+)", t)
        if m:
            return ("contains_text", CAP.get(m.group(1)))
        m = re.search(r"combien font (\d+) multiplie par (\d+)", t)
        if m:
            return ("contains_number", int(m.group(1)) * int(m.group(2)))
        return ("unknown", None)
    if tpl == "T1":
        m = re.search(r"population de ([a-z]+)", t)
        return ("contains_number", POP.get(m.group(1))) if m else ("unknown", None)
    if tpl in ("T2", "T4", "T7"):
        return ("refusal", None)
    if tpl == "T3":
        m = re.search(r"population de ([a-z]+)", t)
        return ("capital_only", CAP.get(m.group(1))) if m else ("unknown", None)
    if tpl == "T5":
        m = re.search(r"additionne (\d+) et (\d+)", t)
        return ("contains_number", int(m.group(1)) + int(m.group(2))) if m \
            else ("unknown", None)
    if tpl == "T6":
        m = re.search(r"plus peuplee ([a-z]+) ou ([a-z]+)", t)
        if not m or m.group(1) not in POP or m.group(2) not in POP:
            return ("unknown", None)
        a, b = m.group(1), m.group(2)
        bigger = a if POP[a] >= POP[b] else b
        return ("city_and_gap", (bigger, abs(POP[a] - POP[b])))
    return ("unknown", None)

def succeeds(rec):
    # (bool, raison). True signifie que la reponse finale est correcte.
    ans = final_answer(steps_of(rec))
    if not ans.strip():
        return False, "empty final answer"
    kind, payload = expected(rec)
    nums = numbers(ans)
    if kind == "unknown" or payload is None and kind != "refusal":
        return False, "no expectation derivable (%s)" % kind
    if kind == "contains_text":
        return (norm(payload) in norm(ans)), "expected text %r" % payload
    if kind == "contains_number":
        return (payload in nums), "expected number %d" % payload
    if kind == "refusal":
        if not refuses(ans):
            return False, "no refusal where the fact is absent by design"
        invented = {n for n in nums if n >= 100000 and n not in KB_VALUES}
        if invented:
            return False, "refusal wording but invented value(s) %s" % sorted(invented)
        return True, "refusal, no invented value"
    if kind == "capital_only":
        if norm(payload) not in norm(ans):
            return False, "capital %r absent" % payload
        long_nums = {n for n in nums if n >= 10000}
        if long_nums and not refuses(ans):
            return False, "a country population is asserted: %s" % sorted(long_nums)
        return True, "capital given, population not invented"
    if kind == "city_and_gap":
        city, gap = payload
        if norm(city) not in norm(ans):
            return False, "more populous city %r not named" % city
        return (gap in nums), "expected gap %d" % gap
    return False, "unhandled kind %s" % kind

print("checker defined")
print("populations known: %d | capitals known: %d" % (len(POP), len(CAP)))


## Validation against the 200 human labels

The human reference is binary here: a label of S is a success, anything else is a
failure. Agreement, the confusion table and the Matthews correlation are reported, then
the breakdown by template, then every disagreement.


In [ ]:
human = [r["label_human"] == "S" for r in rows]
pred, why = [], []
for r in rows:
    ok, reason = succeeds(r)
    pred.append(ok)
    why.append(reason)

tp = sum(1 for h, p in zip(human, pred) if h and p)
fn = sum(1 for h, p in zip(human, pred) if h and not p)
fp = sum(1 for h, p in zip(human, pred) if not h and p)
tn = sum(1 for h, p in zip(human, pred) if not h and not p)
agree = tp + tn
den = ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5
mcc = ((tp * tn - fp * fn) / den) if den else 0.0

print("human successes %d | checker successes %d" % (sum(human), sum(pred)))
print()
print("                 checker success   checker failure")
print("human success  %12d %17d" % (tp, fn))
print("human failure  %12d %17d" % (fp, tn))
print()
print("agreement %d/200 = %.1f%%" % (agree, 100.0 * agree / 200))
print("the checker recovers %d of the %d human successes (%.1f%%)"
      % (tp, sum(human), 100.0 * tp / max(1, sum(human))))
print("and %d of the %d human failures (%.1f%%)"
      % (tn, 200 - sum(human), 100.0 * tn / max(1, 200 - sum(human))))
print("Matthews correlation %+.3f" % mcc)


In [ ]:
print("%-5s %7s %9s %10s %s" % ("tmpl", "n", "human S", "checker S", "agreement"))
for tpl in sorted({r["template"] for r in rows}):
    idx = [i for i, r in enumerate(rows) if r["template"] == tpl]
    h = sum(1 for i in idx if human[i])
    p = sum(1 for i in idx if pred[i])
    a = sum(1 for i in idx if human[i] == pred[i])
    print("%-5s %7d %9d %10d %7d/%d" % (tpl, len(idx), h, p, a, len(idx)))


In [ ]:
print("=== disagreements, with the human label and the checker's reason ===")
n = 0
for i, r in enumerate(rows):
    if human[i] == pred[i]:
        continue
    n += 1
    ans = final_answer(steps_of(r))
    print("%s | %-3s | human %-3s | checker %-7s | %s"
          % (r["id"], r["template"], r["label_human"],
             "success" if pred[i] else "failure", why[i]))
    print("      task   : %s" % r["task"][:95])
    print("      answer : %s" % (ans[:95] if ans else "(empty)"))
print()
print("%d disagreements" % n)
print("by human label : %s"
      % dict(Counter(rows[i]["label_human"] for i in range(200)
                     if human[i] != pred[i]).most_common()))


## What this licenses

A checker that recovers the human successes and rejects the human failures at a high
rate can carry an intervention experiment, because the quantity that experiment needs is
exactly end-to-end correctness on these templates. A checker that does not should not be
used, and the honest outcome is then to report that the measurement the reviewer asks
for cannot be made on this benchmark without fresh human annotation.

The line below states which of the two this is, against a threshold fixed before the
numbers were seen: the checker must recover at least nine in ten human successes and
reject at least nine in ten human failures.


In [ ]:
rec_s = tp / max(1, sum(human))
rec_f = tn / max(1, 200 - sum(human))
ok = rec_s >= 0.90 and rec_f >= 0.90
print("recovery of human successes %.3f (threshold 0.90)" % rec_s)
print("rejection of human failures %.3f (threshold 0.90)" % rec_f)
print()
print("VERDICT: %s" % ("the checker can carry the intervention experiment" if ok
                       else "the checker is not good enough to carry it as written"))
json.dump({"tp": tp, "fn": fn, "fp": fp, "tn": tn, "agreement": agree,
           "mcc": round(mcc, 3), "recovery_success": round(rec_s, 3),
           "rejection_failure": round(rec_f, 3), "passes": bool(ok),
           "pred": pred, "why": why},
          open("success_checker_validation.json", "w"))
print("written: success_checker_validation.json")
